# Experimentación: Optimización de Redes Neuronales

**Resumen:** En esta sección se presentan los distintos experimentos realizados sobre el clasificador de imágenes basado en un MLP.

### Experimento 1: Modelo Base (Dataset Original)
* **MLflow ID:** `cerdo-preocupado-653`

**Configuración:**
* Adam
* Learning Rate: `0.001`
* Batch Size: `32`

**Resultados:**
* Train Accuracy: 62.50%
* Validation Accuracy: 51.66%
* Validation Loss: 1.2158

**Análisis:** El modelo mostró un overfitting severo: aprendía muy bien entrenamiento pero generalizaba mal. Más adelante descubrimos que el resultado estaba contaminado por *Data Leakage*, ya que había imágenes repetidas entre train y validation.

### Experimento 2: Dataset Filtrado
* **MLflow ID:** `yak-gracioso-449`

**Configuración:** Se mantuvo exactamente la misma arquitectura y parámetros:
* Adam
* Learning Rate: `0.001`
* Batch Size: `32`

**Resultados:**
* Train Accuracy: ~62.00%
* Validation Accuracy: 45.27%

**Análisis:** Luego de eliminar imágenes duplicadas y una imagen dañada, el modelo pasó a evaluarse sobre un conjunto de validación nuevo. Aunque las métricas bajaron, este resultado representa una base mucho más confiable para continuar optimizando el modelo.

### Experimento 3: Variamos Learning Rate
**Configuración:** Se mantuvo la misma arquitectura y configuración base, modificando únicamente el learning rate:
* Adam
* Learning Rate: `0.01`, `0.001`, `0.0001`, `0.00001`
* Batch Size: `32`

**Análisis:** Con lr = `0.0001`, el entrenamiento se volvió más estable y el modelo logró generalizar mejor. Se obtuvo la mejor precisión de validación hasta el momento, además de una diferencia menor entre entrenamiento y validación, indicando una reducción del overfitting.

### Experimento 4: Variamos Weight Decay
**Análisis:** Se realizó un experimento modificando el Weight Decay en el optimizador Adam. Los resultados mostraron un empate técnico en la precisión de validación (56.11%) y un levísimo incremento en la precisión de entrenamiento (64.08%). Sin embargo, la función de pérdida en validación aumentó de 1.1333 a 1.1602. Nos quedamos con `Weight Decay = 0.0001`.

### Experimento 5: Batch Size
**Configuración:** Se mantuvo la arquitectura y el learning rate elegidos previamente, modificando únicamente el batch size:
* Adam
* Learning Rate: `0.0001`
* Batch Size: `64`, `32`, `128`

**Análisis:** El cambio de batch size de `32` a `64` no produjo mejoras en el rendimiento del modelo. Al aumentar el batch size a `128`, el rendimiento del modelo volvió a empeorar. Tanto la precisión de entrenamiento como la de validación fueron menores respecto a los experimentos con batch sizes más chicos.

Luego de comparar los distintos valores probados, se decidió mantener `Batch Size = 32`, ya que obtuvo la mejor precisión de validación y la menor pérdida.

### Experimento 6: Optimizador RMSprop
* **Configuración:** Se mantuvo la estructura regularizada óptima, modificando únicamente el algoritmo del optimizador para contrastarlo con la corrida previa.
* Optimizador RMSprop
* Learning Rate = `0.0001`
* Batch Size = `32`
* Weight Decay = `1e-4`

**Análisis:** A pesar del leve incremento en el Accuracy de validación, la función de pérdida (Validation Loss) se encareció notablemente, subiendo a 1.2103.

### Experimento 7: Optimizador SGD con Momentum
* **Configuración:** Estructura regularizada base (MLP).
* Optimizador SGD
* Learning Rate = `0.001`
* Momentum = `0.9`
* Batch Size = `32`
* Weight Decay = `1e-4`

**Métricas Registradas:**
* Train Accuracy: 57.47%
* Validation Accuracy: 48.33%
* Validation Loss: 1.2380

**Análisis:** La incorporación de SGD con un Learning Rate intermedio de 0.001 no logró igualar el rendimiento alcanzado por los optimizadores adaptativos previos. El modelo experimentó un freno en su velocidad de convergencia, alcanzando un Train Accuracy de apenas 57.47%.

**Diagnóstico Crítico:** Con una caída en la precisión de validación al 48.33% y una Loss que se estancó en 1.2380, se concluye que Adam (`percha-capaz-498`) sigue siendo la configuración óptima para la naturaleza de este set de datos y los límites de cómputo establecidos.

Al explorar el comportamiento de SGD, se realizaron variaciones estratégicas elevando el Momentum a 0.99 e incrementando el Weight Decay a 1e-3. Si bien esta sintonización optimizó notablemente la velocidad de convergencia y mejoró los resultados previos de este algoritmo (alcanzando un 51.67% de precisión en validación), no logró destronar a Adam (`percha-capaz-498`), el cual se mantiene como la opción más eficiente y con menor pérdida general del proyecto.

## Experimento 8: Introducción de BatchNorm y Regularización por Dropout

**Configuración Base:** Se mantuvo el optimizador óptimo Adam ($lr=0.0001$, $batch=32$) y la arquitectura de dos capas ocultas ($512 \rightarrow 128$). El objetivo de esta serie de pruebas fue acelerar la convergencia mediante Normalización de Lote (BatchNorm) y controlar el sobreajuste variando la intensidad del Dropout.

Tras evaluar diferentes tasas y configuraciones de Dropout, la mejor combinación fue aplicar una estrategia de **Regularización Focalizada (Dropout de 0.25 en la primera capa y 0.00 en la segunda)** bajo la ejecución `gusano-aventurero-705`.

Como resultado directo, esta variante logró el **récord histórico del proyecto con un 61.11% de precisión en validación** y un marcado descenso en la pérdida de validación ($1.2186$), demostrando ser el único diseño capaz de maximizar la potencia de BatchNorm sin desestabilizar la generalización de la red.

## Experimento 9: Optimización de la Arquitectura (Capas y Unidades/Capa)

**Configuración Base:** Tras consolidar la estrategia de Dropout Focalizado (0.25 en Capa 1 y 0.00 en Capa 2), el optimizador Adam ($lr=0.0001$) y BatchNorm, se procedió a evaluar el impacto del tamaño de las capas ocultas para hallar el balance óptimo entre capacidad de cómputo y generalización. 

Para esto, se exploraron estructuras más robustas expandiendo las dimensiones a un diseño de punto medio de $768 \rightarrow 192$ (bajo la ejecución `elegante-delfín-973`) y a una red ancha de máxima capacidad de $1024 \rightarrow 256$ (bajo la ejecución `spiffy-cub-831`). 

**Motivo y Análisis Técnico:**
Los resultados revelaron un comportamiento matemático muy claro respecto a la escala del modelo. A medida que incrementamos el número de neuronas por capa, la red redujo drásticamente su función de pérdida en validación, logrando con el modelo de 1042 unidades un mínimo histórico en el proyecto ($1.1182$). Esto demuestra que arquitecturas más grandes optimizan mejor el margen de error general y realizan predicciones con mayor certeza estadística.

Sin embargo, a pesar de la notable mejoría en la pérdida, las pruebas de $768$ y $1024$ unidades quedaron por debajo en precisión pura, registrando un $57.22\%$ y $58.33\%$ de *Accuracy* en validación respectivamente. El incremento masivo de parámetros libres en estos diseños reactivó sutilmente la tendencia de la red a memorizar ruido o patrones específicos del set de entrenamiento. 

Por lo tanto, la estructura compacta original de **$512 \rightarrow 128$ (`gusano-aventurero-705`) se consolida definitivamente como el estándar y campeón del proyecto**, al ser el único diseño capaz de retener el pico máximo de 61.11% de precisión mediante una generalización eficiente.

## Experimento 11: Introducción de Tercera Capa Oculta

**Configuración:** Se testeó aumentar la profundidad de la red. Se configuraron 3 capas ocultas en la ejecución `paloma-útil-954`. Se mantuvo la regularización focalizada (Dropout de 0.25 únicamente en la primera capa) y las 10 épocas de entrenamiento rígidas para garantizar una comparación directa y justa.

Los resultados empíricos demostraron de forma contundente que la profundidad es contraproducente para este tipo de red densa (MLP) aplicada a imágenes vectorizadas. El modelo sufrió un claro efecto de subajuste estructural (*underfitting*): la precisión de entrenamiento cayó al 64.66% y la pérdida escaló a 1.2175, lo que evidencia que a los gradientes les costó optimizar los pesos eficientemente a lo largo de tres bloques de transformación secuencial. Como consecuencia, el *Accuracy* en validación se desplomó al 55.00%.

Este experimento convalida empíricamente que **más capas no implican un mejor rendimiento**. Tras analizar el impacto del ancho (Experimento 9) y de la profundidad (Experimento 11), se concluye de forma definitiva que la arquitectura de **dos capas ocultas ($512 \rightarrow 128$) con Dropout focalizado de `gusano-aventurero-705` es la estructura óptima, balanceada y campeona absoluta del proyecto.**

## Experimento 12: Optimización del Tiempo de Entrenamiento (Escalabilidad de Épocas)

**Configuración Base:** Habiendo consolidado la arquitectura óptima de dos capas ocultas ($512 \rightarrow 128$) con regularización asimétrica (Dropout de 0.25 en la primera capa y 0.00 en la segunda), optimizador Adam ($lr=0.0001$) y BatchNorm, se procedió a evaluar si el modelo requería mayor tiempo de cómputo para alcanzar la convergencia óptima. Se extendió el límite de entrenamiento de 10 a **15 épocas** bajo la ejecución `alce-intrigado-66`.

**Resultados:**
* **Train Accuracy:** 73.56% 📈
* **Validation Accuracy:** **61.66%** 👑 *(Máximo Histórico del Proyecto)*
* **Validation Loss:** **1.1305** 📉 *(Optimización del Margen de Error)*

**Análisis Técnico y Cierre del Laboratorio:**
Este experimento definitivo arrojó el rendimiento más alto de todo el proceso de ingeniería de hiperparámetros. Al extender el entrenamiento a 15 épocas, el modelo demostró que aún contaba con margen de optimización (*nafta en el carretel*) para ajustar los pesos finos de la red, elevando la precisión de entrenamiento al 73.56% sin caer en sobreajuste estricto. 

La combinación del aumento de épocas con la barrera de protección del Dropout del 25% en la primera capa permitió que el *Accuracy* de validación tocara su techo histórico con un **61.66%**, acompañado de un marcado descenso en la función de pérdida general ($1.1305$). 

Este resultado valida de forma empírica la estrategia secuencial del laboratorio: primero se estabilizó el ritmo de aprendizaje, luego se blindó la estructura física contra el ruido, y finalmente se le otorgó el tiempo necesario para consolidar las abstracciones visuales. **El modelo `alce-intrigado-66` queda consagrado como el Campeón Absoluto y Definitivo del proyecto.**

### Conclusiones de Ingeniería de Hiperparámetros y Validación Teórica

Para dar cierre al laboratorio, se contrastaron los resultados empíricos con las recomendaciones de diseño arquitectónico de Yoshua Bengio y Andrew Ng detalladas en la literatura:

1. **Compensación de Varianza mediante Regularización Focalizada:** Tal como indica la teoría de regularización, los modelos con alta densidad de parámetros sufren de Alta Varianza (Overfitting). Al evaluar estructuras anchas como `spiffy-cub-831` ($1024 \rightarrow 256$), se observó este fenómeno. Sin embargo, la implementación de un Dropout del 25% en la primera capa (zona de mayor volumen de parámetros libres) actuó como un filtro MAP (Maximum A Posteriori), reduciendo la varianza y permitiendo estirar el entrenamiento a 15 épocas en `alce-intrigado-66` para consolidar un óptimo histórico de **61.66% de Accuracy**.

2. **El Peligro de la Profundidad Innecesaria (Alto Bias):** El Experimento 11 (`paloma-útil-954`) demostró que añadir profundidad mediante una tercera capa de transformación ($512 \rightarrow 256 \rightarrow 64$) indujo un problema de Alto Bias (Underfitting Structural). Al complejizar el flujo del gradiente y reducir las unidades finales, la red perdió capacidad de representación, aumentando la pérdida de train a un alto 1.2175.

3. **Incompatibilidad de Arquitectura Compleja con Datasets Compactos:** Al no disponer de un volumen masivo de datos (Big Data), la estrategia recomendada por Bengio de mantener arquitecturas controladas con inicializaciones estables (como BatchNorm de forma secuencial) probó ser superior a la mera adición de capas lineales densas.